# **MindEase: AI Engineering Model Development (Quest Completion Edition)**
**Project:** Healthy Lives & Well-being  

### **Checklist Progres AI Engineer:**
1. [x] **Main Quest:** TensorFlow Functional API (Multi-Output)
2. [x] **Main Quest:** Custom Callback Implementation
3. [x] **Main Quest:** Model Export (.keras & .tflite)
4. [x] **Side Quest:** TensorBoard Monitoring
5. [x] **Side Quest:** GenAI Recommendations Simulation

### **1. Setup & Preprocessing**

In [22]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import datetime
import os

# Preprocessing
df = pd.read_csv('mental_health_clean.csv').dropna(subset=['risk_level', 'burnout_score'])
le_gender = LabelEncoder()
df['gender'] = le_gender.fit_transform(df['gender'])
le_risk = LabelEncoder()
df['risk_level'] = le_risk.fit_transform(df['risk_level'])
X = df.drop(columns=['burnout_score', 'risk_level', 'dropout_risk', 'mental_health_index'])
y_class = df['risk_level']
y_reg = df['burnout_score']
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X_scaled, y_class, y_reg, test_size=0.2, random_state=42
)

print("Dataset siap untuk integrasi!")

Dataset siap untuk integrasi!


### **2. Multi-Output Model (Functional API)**

In [23]:
def build_mindease_model(input_shape):
    inputs = layers.Input(shape=(input_shape,), name='main_input')
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)

    # Branch 1: Risk Classification
    class_output = layers.Dense(3, activation='softmax', name='risk_output')(x)

    # Branch 2: Burnout Regression
    reg_output = layers.Dense(1, name='burnout_output')(x)

    return models.Model(inputs=inputs, outputs=[class_output, reg_output])

model = build_mindease_model(X_train.shape[1])
model.compile(
    optimizer='adam',
    loss={'risk_output': 'sparse_categorical_crossentropy', 'burnout_output': 'mse'},
    metrics={'risk_output': 'accuracy', 'burnout_output': 'mae'}
)

### **3. Callbacks & TensorBoard**

In [24]:
class MindEaseCallback(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('val_risk_output_accuracy') >= 0.85:
            print(f"\nTarget Akurasi 85% tercapai. Model siap di-deploy!")
            self.model.stop_training = True

log_dir = os.path.join("logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tb_callback = callbacks.TensorBoard(log_dir=log_dir)
mindease_callback = MindEaseCallback()

### **4. Training & Export**

In [25]:
model.fit(
    X_train, {'risk_output': y_class_train, 'burnout_output': y_reg_train},
    validation_data=(X_test, {'risk_output': y_class_test, 'burnout_output': y_reg_test}),
    epochs=50, batch_size=32, callbacks=[mindease_callback, tb_callback]
)

# Export
model.save('model_mindease_final.keras')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
with open('model_mindease_final.tflite', 'wb') as f:
    f.write(converter.convert())
print("Model siap diserahkan ke tim Full-Stack!")

Epoch 1/50
2134/2138 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - burnout_output_loss: 1.1893 - burnout_output_mae: 0.8219 - loss: 1.6514 - risk_output_accuracy: 0.8005 - risk_output_loss: 0.4621
Target Akurasi 85% tercapai. Model siap di-deploy!
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 26s 9ms/step - burnout_output_loss: 0.9684 - burnout_output_mae: 0.7569 - loss: 1.3433 - risk_output_accuracy: 0.8399 - risk_output_loss: 0.3749 - val_burnout_output_loss: 0.7664 - val_burnout_output_mae: 0.6640 - val_loss: 1.0762 - val_risk_output_accuracy: 0.8675 - val_risk_output_loss: 0.3092
Saved artifact at '/tmp/tmpd5oh53c2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 16), dtype=tf.float32, name='main_input')
Output Type:
  List[TensorSpec(shape=(None, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)]
Captures:
  134621548130576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134621548131152: TensorSpe

### **5. GenAI Prediction Simulation (Side Quest)**

In [26]:
def mindease_ai_assistant(sample_data):
    # 1. Prediksi Model
    res = model.predict(np.array([sample_data]))
    risk_idx = np.argmax(res[0])
    risk_label = le_risk.inverse_transform([risk_idx])[0]
    score = res[1][0][0]

    # 2. Simulasi GenAI Recommendation
    print(f"[MindEase Prediction]: {risk_label} Risk (Score: {score:.2f})")
    print("--- GenAI Recommendation ---")
    if risk_label == 'High':
        print("Gemini: 'Sepertinya kamu sedang sangat lelah. Prioritaskan istirahat hari ini dan hubungi profesional jika butuh bantuan.'")
    else:
        print("Gemini: 'Kondisimu stabil. Tetap jaga keseimbangan antara belajar dan waktu luang ya!'")

mindease_ai_assistant(X_test[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step
[MindEase Prediction]: Low Risk (Score: 2.74)
--- GenAI Recommendation ---
Gemini: 'Kondisimu stabil. Tetap jaga keseimbangan antara belajar dan waktu luang ya!'
